# Stage 06: Extra Credit - Robustness & Error Propagation

## Overview
This notebook evaluates how perturbations (heavy blur, crushed low contrast) propagate through the hybrid pipeline and demonstrates automated quality gating.


In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import sys
from pathlib import Path
import torch
import pandas as pd
import matplotlib.pyplot as plt

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name.lower() == "notebooks" else CURRENT_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.unet import SmallUNet
from src.pipeline import run_robustness_experiment


## 1. Evaluate Corrupted Image Variants


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SmallUNet(features=(16, 32, 64, 128)).to(device)

ckpt_path = PROJECT_ROOT / "outputs" / "models" / "unet_combined_bce___dice.pth"
if ckpt_path.exists():
    state = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(state["model_state_dict"])

data_dir = PROJECT_ROOT / "data" / "nuclei_dataset"
if not data_dir.exists():
    data_dir = PROJECT_ROOT / "nuclei_dataset"

clean_imgs = [
    data_dir / "test" / "images" / "test_000.png",
    data_dir / "test" / "images" / "test_004.png",
]
corrupt_imgs = [
    data_dir / "test_corrupted" / "images" / "test_000_blur.png",
    data_dir / "test_corrupted" / "images" / "test_000_lowcontrast.png",
    data_dir / "test_corrupted" / "images" / "test_004_blur.png",
    data_dir / "test_corrupted" / "images" / "test_004_lowcontrast.png",
]

robust_df = run_robustness_experiment(
    model=model,
    clean_image_path=clean_imgs,
    corrupted_image_paths=corrupt_imgs,
    llm_model="llama3.2",
    device=device
)
display(robust_df[["image_id", "condition", "mean_intensity", "intensity_range", "connected_component_count", "area_fraction", "quality_gate", "quality_reason", "llm_quality_flag"]])


,image_id,condition,mean_intensity,intensity_range,connected_component_count,area_fraction,quality_gate,quality_reason,llm_quality_flag
0,test_000,clean,0.0177,0.3468,8,0.0237,pass,Image passed deterministic quality checks.,pass
1,test_004,clean,0.0666,0.3468,42,0.2183,pass,Image passed deterministic quality checks.,pass
2,test_000_blur,blur,0.0159,0.1836,7,0.0391,pass,Image passed deterministic quality checks.,pass
3,test_000_lowcontrast,lowcontrast,0.4256,0.0505,1,1.0000,reject,Foreground coverage exceeds 75% (threshold col...,rejected_pre_llm
4,test_004_blur,blur,0.0643,0.2847,15,0.3751,pass,Image passed deterministic quality checks.,pass
5,test_004_lowcontrast,lowcontrast,0.4330,0.0505,1,1.0000,reject,Foreground coverage exceeds 75% (threshold col...,rejected_pre_llm
